# Stent and artery

The stent is tied into a generated test artery with meshtying. This is the only one of the
three simulation types that involves an artery at all, and it is where beam-to-solid
meshtying lives rather than contact.

It is a smoke test of the whole chain and not the physics of the reference papers. The
artery wall uses a placeholder material, the coupling is tied rather than true contact,
and the balloon is replaced by a radial point force at each stent node. So it writes a
runnable input and a coupling report, and it has never been solved. There is no result
here to read. For deployment physics use `simulation_stent_and_balloon.ipynb`, which is
driven only through contact and has converged runs behind it.

In [1]:
# "sphinx_gallery" renders each Plotly figure as a self-contained text/html output,
# so the views below also work on the documentation website without a running kernel.
import plotly.io as pio
pio.renderers.default = "sphinx_gallery"

## 2. Load the stent

The stent has to be skeletonised already, which is what `stent_skeleton.ipynb` does.
Everything below is sized from the measurements in that output, so the stent is the
only input.

In [2]:
from pathlib import Path

from stentfit import Artery, Simulation, Stent
from stentfit.sim import StentArterySettings

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

STENT_NAME = "stent01"
STENT_DIR = REPO / "examples/data/output/stent_skeleton" / STENT_NAME
OUTPUT_DIR = REPO / "examples/data/output/simulation"

stent = Stent.load(str(STENT_DIR), stent_name=STENT_NAME)

[resume] loading point cloud from /Users/vural/Desktop/RWTH - SiSc/Projects/Github/stentFIT/examples/data/output/stent_skeleton/stent01/ring_points.csv ...
[resume] restored 10 per-ring 2D skeletons + 1,824,400 surface points. Ready for the manual-edit step.


## 3. Parameters

The artery is sized from the stent, so only its shape and its wall are set here. The lumen
radius is the stent's outer radius plus `inner_margin`, and the length is a multiple of the
stent length, so the clamped ends sit clear of the stent.

Both element sizes are relative to the strut thickness, and it is the ratio between them
that the coupling check tests.

In [3]:
artery = Artery(
    stent,
    artery_type="curved",      # "straight" | "curved" | "s_bend"
    inner_margin=0.5,          # extra clearance [mm] between stent and artery inner wall
    wall_thickness=0.5,        # artery wall thickness [mm] (0 = lumen surface only)
    noise_amplitude=0.05,      # fractional wall roughness (0 = smooth pipe)
    noise_seed=0,
    bend_angle_deg=180.0,      # only used by "curved" / "s_bend"
    mesh_type="HEX8",          # "TET4" | "TET10" | "HEX8"
    artery_youngs=2.0,         # wall Young's modulus [MPa]
)

sa = StentArterySettings(
    # Stent beam material
    material="elastic",
    youngs=2.0e5,
    poisson=0.3,
    density=0.0,
    beam_class="Beam3rHerm2Line3",

    # Element sizing, both relative to the stent's strut thickness:
    #   solid element = strut * factor_solid
    #   beam element  = strut * factor_solid * factor_beam
    factor_solid=1.5,
    factor_beam=1.2,

    # Loading: a radial outward point force per stent node, standing in for the balloon
    expansion_force=1e-4,      # N
    n_steps=10,                # quasi-static steps, force ramps 0 -> full
)

Artery type      : curved
Artery radius    : 2.072 mm (lumen)
Wall thickness   : 0.500 mm  (outer radius 2.572 mm)
Noise amplitude  : 0.05 (5% of radius)  seed=0
Bend angle       : 180.0 deg
Bend radius      : 7.78 mm  (arc = 24.43 mm)
Arc length       : 27.14 mm  (stent 18.09 mm = 67% of artery)
Centreline       : 150 points  bounds [0. 0. 0.] → [15.55  0.    9.13]
Mesh             : 19,204 vertices  38,400 faces  watertight=True


## 4. Build the input

One call meshes the artery wall as a 3D solid with GMSH, warps the stent onto the artery
centreline, ties the beams to the lumen surface, checks the coupling, and writes the input.

The coupling check is the point of this notebook. It compares the stiffness of the beam
against the solid and the two element sizes against each other, following Steinbrecher et
al., and the build refuses to write an input that violates them. An input file that breaks
the mixed-dimensional assumptions should not look runnable.

In [4]:
sim = Simulation(stent, sim_type="stent_artery", settings=sa,
                 artery=artery, output_dir=OUTPUT_DIR)

written = sim.build_input()
sim.check()

stent stent01: diameter 3.152 mm, strut 0.1073 mm
  sizing: solid 0.1609 mm, beam 0.1931 mm

Meshed 135/135 curves (0 skipped) into a 1D beam mesh:
  2,167 nodes, 1,016 Beam3rHerm2Line3 elements
  cross-section radius 0.0536 mm, target element length 0.1931 mm
[saved] stent_warped.4C.yaml
[gmsh] artery wall meshed (HEX8): r_inner=2.072 r_outer=2.572 arc length=27.140 mm  (element size 0.161 mm)
[gmsh] 59,842 nodes, 44,616 HEX8 elements  noise=0.05, warped onto centreline
[gmsh] surface node sets: lumen=14960 inlet=352 outlet=352
[saved] /Users/vural/Desktop/RWTH - SiSc/Projects/Github/stentFIT/examples/data/output/simulation/stent_artery/stent01/artery_solid.4C.yaml
mixed-dimensional coupling (Steinbrecher et al.):
  [ok  ] solid element >= beam diameter, solid    0.1609  (>= 0.1073)
  [ok  ] shortest beam / solid element            1.1183  (>= 1)
  [ok  ] longest beam / solid element             1.3820  (<= 6 optimal, <= 8 valid)
  [ok  ] beam / solid stiffness                100000.0

[{'rules': [{'name': 'solid element >= beam diameter, solid',
    'value': 0.1608844107649163,
    'limit': '>= 0.1073',
    'passed': True,
    'optimal': True},
   {'name': 'shortest beam / solid element',
    'value': 1.118334240964613,
    'limit': '>= 1',
    'passed': True,
    'optimal': True},
   {'name': 'longest beam / solid element',
    'value': 1.3820415738871334,
    'limit': '<= 6 optimal, <= 8 valid',
    'passed': True,
    'optimal': True},
   {'name': 'beam / solid stiffness',
    'value': 100000.0,
    'limit': '>= 10',
    'passed': True,
    'optimal': True}],
  'all_passed': True,
  'all_optimal': True}]

## 5. What came out

In [5]:
record = sim.built[0]["record"]

print(f"beam elements  : {record['beam_model']['n_elements']:,}")
print(f"beam element   : {record['beam_model']['target_element_length_mm']:.4f} mm")
print(f"artery         : {record['artery']['type']}, "
      f"r = {record['artery']['radius_mm']:.3f} mm, "
      f"L = {record['artery']['length_mm']:.3f} mm")
print(f"artery solid   : {sim.artery.solid_yaml}")
print(f"coupling       : {record['coupling_method']}")

for path in written:
    print(f"\n{path.name}")
    for f in sorted(path.parent.iterdir()):
        print(f"   {f.name:38s} {f.stat().st_size / 1e6:7.2f} MB")

beam elements  : 1,016
beam element   : 0.1931 mm
artery         : curved, r = 2.072 mm, L = 27.140 mm
artery solid   : /Users/vural/Desktop/RWTH - SiSc/Projects/Github/stentFIT/examples/data/output/simulation/stent_artery/stent01/artery_solid.4C.yaml
coupling       : beam-to-solid surface meshtying (tied), mortar line2

simulation.4C.yaml
   artery_solid.4C.yaml                      8.09 MB
   artery_stent.4C.yaml                      8.58 MB
   artery_stent_mesh_beam.vtu                0.28 MB
   artery_stent_mesh_solid.vtu               6.20 MB
   run_parameters.yaml                       0.00 MB
   runs_summary.csv                          0.00 MB
   simulation.4C.yaml                        8.84 MB
   simulation.nox.xml                        0.00 MB
   stent_warped.4C.yaml                      0.42 MB


## 6. Where the files are

```
examples/data/output/simulation/stent_artery/<stent>/
    artery_solid.4C.yaml     the artery wall, meshed by GMSH
    stent_warped.4C.yaml     the stent, warped onto the artery centreline
    artery_stent.4C.yaml     the two tied together
    run_parameters.yaml      every parameter that produced this build
```

Open the `.vtu` mesh previews in ParaView to see the stent sitting inside the artery.
Solving this is possible with `python -m stentfit.run solve stent_artery <stent>`, but it
has not been done, and with a placeholder wall material the result would not mean much.